In [0]:
import os
import pytest
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

@pytest.fixture(scope="session")
def spark_env():
    # Cria um ambiente Spark dentro do próprio GitHub Actions
    spark-session = SparkSession.builder \
        .master("local[*]") \
        .appName("DataOps-Testing") \
        .getOrCreate()

    env = os.getenv("DATABRICKS_ENVIRONMENT", "dev").strip().lower()
    target_catalog = f"ecommerce_{env}"

    return {
        "spark": spark_session,
        "env" : env,
        "catalog": target_catalog,
        "bronze_schema": f"{target_catalog}.bronze",
        "silver_schema": f"{target_catalog}.silver",
        "gold_schema": f"{target_catalog}.gold"
    }

def test_quarentena_registros_nuls(spark_env):
    # Cria uma massa de dados fake contendo um erro intencional (ID nulo)        
    spark = spark_env["spark"]
    catalog_atual = spark_env["catalog"]

    schema = StructType([
        StructField("transaction_id", StringType(), True),
        StructField("user_id", StringType(), True)
        ])
    dados_teste = [("TX_12345", "USR_1"), (None, "USR_2")] # o 2º registro deve ir pra quarentena
    df = spark.createDataFrame(dados_teste, schema)
    
    # Aplica a regra de validação
    df_trusted = df.filter(df.transaction_id.isNotNull())
    df_quarentena = df.filter(df.transaction_id.isNull())
    
    # Asserções
    assert df_trusted.count() == 1
    assert df_quarentena.count() == 1
    assert spark_env["env"] in ["dev", "staging", "prod"], "Erro: Ambiente configurado é desconhecido."    
    
    